[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/geraldmc/irilab2026/blob/main/notebooks/r1/r1-q2/00b_matrix_quality_check.ipynb)

# R1-Q2 Week 1 (continued) — Matrix quality check

This notebook inspects the filtered shoot and root expression matrices
that `00_question_orientation.ipynb` produced. It does not change them,
re-filter them, or write anything to disk — it reads what's there and
reports back.

The point: a matrix can pass each per-step check in N0 (the log
transform worked, the MAD distribution looks right, the filtered shapes
are plausible) and still not be in shape for downstream co-expression
work. The diagnostics here are the next layer of "is this fit for
purpose" — sample-level structure, gene-level constancy, control
probes, and the whole-matrix correlation picture.

By the end of this notebook you will have:

- A per-condition sample count showing whether any tissue × stress cell
  is thinly populated.
- A sample-level PCA per tissue showing whether any samples sit far
  from the rest of their condition (outlier candidates).
- A pairwise sample-correlation heatmap per tissue showing whether
  replicates and conditions cluster as expected.
- A flag for genes that survived the MAD filter but have effectively
  zero range across samples.
- A count of control probes (`AFFX-*`, including the bacterial spike-in
  controls) that survived the filter, with a named decision for you to
  make.
- A four-number summary of the pairwise gene-gene correlation
  distribution per tissue, and a synthesis cell that names any concerns
  and points to next steps.

If the diagnostics come back clean, you proceed to Week 2's
`01_wgcna.ipynb`. If they surface a problem, you go back to N0, revise
the relevant pre-commit and re-run, then re-run this notebook to
confirm the fix. The synthesis at the end of this notebook is where
that branch decision gets made.

In [ ]:
# There are three patterns for installing irilab2026 from GitHub, depending on your needs. 

# The first is for a fresh, complete runtime. This installs irilab2026 and every dependency it declares. 
# It skips anything pip already sees as installed. This is ideal for a new environment or when you want 
# to be sure everything is up to date.

#!pip install git+https://github.com/geraldmc/irilab2026.git@main
 
# The second is for code iteration only. It reinstalls irilab2026 itself, ignoring its dep list entirely. 
# The dep tree is left exactly as it is. This is ideal for iterating on irilab2026 itself, when you know 
# the deps are already satisfied and don't want to waste time reinstalling them.
# 
!pip install --force-reinstall --no-deps git+https://github.com/geraldmc/irilab2026.git@main

# The third is for developers who want to work with the code directly and see changes reflected 
# immediately without reinstalling.

#!pip install --force-reinstall --no-deps git+https://github.com/geraldmc/irilab2026.git@main
#!pip install git+https://github.com/geraldmc/irilab2026.git@main

In [ ]:
# Mount Drive, set up irilab2026, point to the R1-Q2 output directory.
import irilab2026 as iri
iri.setup()

OUTPUT_DIR = iri.output_dir("r1_q2")
print(f"Output directory: {OUTPUT_DIR}")

## 2) Load and orient

Three pieces of input feed every diagnostic that follows:

1. **The two filtered parquets** (`shoot_filtered.parquet`,
   `root_filtered.parquet`) — what N0 wrote at the end of its Section
   3.
2. **The sample metadata** — which GSM belongs to which tissue, which
   stress condition, and which replicate. Comes from
   `iri.atgenexpress_metadata()`, the same function N0 used.
3. **`precommit.json`** — your Week 1 pre-commits. We don't act on
   these here, but we'll echo the `gene_filtering` block so the rest
   of the notebook has the filter choices visible on screen.

Load each in turn, then confirm the sample columns in the two
parquets line up with the metadata's GSM index. If any of these
loads fail, the error message tells you what to do.

In [ ]:
import pandas as pd

shoot_path = OUTPUT_DIR / "shoot_filtered.parquet"
root_path = OUTPUT_DIR / "root_filtered.parquet"

try:
    shoot = pd.read_parquet(shoot_path)
    root = pd.read_parquet(root_path)
except FileNotFoundError as exc:
    raise FileNotFoundError(
        f"\nCould not find a filtered matrix.\n"
        f"  Looked for: {shoot_path}\n"
        f"              {root_path}\n\n"
        f"Run `00_question_orientation.ipynb` to completion first — "
        f"its Section 3 writes both parquets to this location.\n"
    ) from None

print(f"Shoot matrix: {shoot.shape[0]:>5} genes × {shoot.shape[1]:>3} samples")
print(f"Root matrix:  {root.shape[0]:>5} genes × {root.shape[1]:>3} samples")